# Data Flywheel — Exploration

Inspect raw inference logs, understand filter drop rates, and explore quality score distributions.

**Prerequisites:** Services running (`make up`), logs seeded (`make seed`)

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from elasticsearch import Elasticsearch
from pymongo import MongoClient
from dotenv import load_dotenv

load_dotenv()

ES_HOST = os.getenv('ES_HOST', 'http://localhost:9200')
MONGO_URI = os.getenv('MONGO_URI', 'mongodb://localhost:27017')
MONGO_DB = os.getenv('MONGO_DB', 'data_flywheel')

es = Elasticsearch(ES_HOST)
mongo = MongoClient(MONGO_URI)
db = mongo[MONGO_DB]

print('Connected to Elasticsearch:', es.ping())
print('Collections:', db.list_collection_names())

## 1. Raw Inference Logs

In [ ]:
resp = es.search(index='inference_logs', body={'query': {'match_all': {}}, 'size': 1000})
logs = [hit['_source'] for hit in resp['hits']['hits']]
df_logs = pd.DataFrame(logs)

print(f'Total logs: {len(df_logs)}')
df_logs.head()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Latency distribution
df_logs['latency_ms'].hist(bins=30, ax=axes[0], color='steelblue', edgecolor='white')
axes[0].set_title('Latency Distribution (ms)')
axes[0].set_xlabel('Latency (ms)')

# Model distribution
df_logs['model'].value_counts().plot(kind='bar', ax=axes[1], color='steelblue')
axes[1].set_title('Logs by Model')
axes[1].tick_params(axis='x', rotation=30)

# Response length
df_logs['response'].str.len().hist(bins=30, ax=axes[2], color='steelblue', edgecolor='white')
axes[2].set_title('Response Length Distribution')
axes[2].set_xlabel('Characters')

plt.tight_layout()
plt.show()

## 2. Filter Drop Rates

In [ ]:
import sys
sys.path.insert(0, '..')

from orchestrator.services.curator.filters import FilterPipeline

filter_pipeline = FilterPipeline(
    min_quality_score=0.7,
    max_token_length=2048,
    min_response_length=20,
    remove_pii=True,
)

filtered = filter_pipeline.run(logs)
dropped = len(logs) - len(filtered)

print(f'Before filter: {len(logs)}')
print(f'After filter:  {len(filtered)}')
print(f'Dropped:       {dropped} ({dropped/len(logs)*100:.1f}%)')

In [ ]:
# Visualise drop reasons
summary = filter_pipeline.run(logs, return_summary=True) if hasattr(filter_pipeline, 'return_summary') else {}

stages = ['raw', 'after_filter']
counts = [len(logs), len(filtered)]

plt.figure(figsize=(8, 4))
bars = plt.bar(stages, counts, color=['#e74c3c', '#2ecc71'], edgecolor='white')
for bar, count in zip(bars, counts):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 5,
             str(count), ha='center', fontweight='bold')
plt.title('Filter Pipeline — Sample Counts')
plt.ylabel('Samples')
plt.tight_layout()
plt.show()

## 3. Quality Score Distribution

In [ ]:
from orchestrator.services.curator.filters import compute_quality_score

scores = [compute_quality_score(log.get('response', '')) for log in logs]
df_scores = pd.DataFrame({'quality_score': scores})

plt.figure(figsize=(10, 4))
plt.hist(scores, bins=30, color='steelblue', edgecolor='white')
plt.axvline(0.7, color='red', linestyle='--', label='Min threshold (0.7)')
plt.title('Quality Score Distribution')
plt.xlabel('Quality Score')
plt.ylabel('Count')
plt.legend()
plt.tight_layout()
plt.show()

print(df_scores.describe())

## 4. Curated Datasets

In [ ]:
datasets = list(db.datasets.find({}, {'samples': 0}))
df_datasets = pd.DataFrame(datasets)
df_datasets['_id'] = df_datasets['_id'].astype(str)
print(f'Total curated datasets: {len(df_datasets)}')
df_datasets[['_id', 'run_id', 'sample_count', 'created_at']].tail(10)